In [17]:
import numpy as np
from collections import Counter

# Create a corpus of text data
corpus = [
	"i like deep learning",
	"i like natural language processing",
	"deep learning is fun",
]

# Tokenize the corpus into words
tokens = [word for sentence in corpus for word in sentence.split()]

# Count + build vocab
word_counts = Counter(tokens)
vocab = sorted(word_counts.keys())

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}

vocab_size = len(vocab)
print("Vocabulary Size:", vocab_size)
for word, idx in word2idx.items():
	print(f"Word: {word}, Index: {idx}")

token_ids = [word2idx[word] for word in tokens]
print("Token IDs:", token_ids)

Vocabulary Size: 9
Word: deep, Index: 0
Word: fun, Index: 1
Word: i, Index: 2
Word: is, Index: 3
Word: language, Index: 4
Word: learning, Index: 5
Word: like, Index: 6
Word: natural, Index: 7
Word: processing, Index: 8
Token IDs: [2, 6, 0, 5, 2, 6, 7, 4, 8, 0, 5, 3, 1]


In [18]:
# generate trainting data
def generate_training_pairs(token_ids, window_size=2):
	"Return training data for skip-gram model"
	pairs = []
	for center_pos in range(window_size, len(token_ids) - window_size):
		center = token_ids[center_pos]
		for j in range(-window_size, window_size + 1):
			if j == 0:
				continue
			context = token_ids[center_pos + j]
			pairs.append((center, context))
	return pairs

pairs = generate_training_pairs(token_ids, window_size=2)
print(pairs[:5])

[(0, 2), (0, 6), (0, 5), (0, 2), (5, 6)]


In [19]:
# initialize parameters
embed_dim = 5

# randomly init center word vectors and context word vectors
V = np.random.randn(vocab_size, embed_dim) * 0.01 # (V, d)
U = np.random.randn(vocab_size, embed_dim) * 0.01 # (V, d)

print(V.shape)
print(V[:3])
print(U.shape)
print(U[:3])

(9, 5)
[[-0.00186269 -0.00281597  0.00625944 -0.00197933 -0.01211832]
 [ 0.0031928   0.00462373  0.01116926  0.02059378  0.01241338]
 [ 0.00530571 -0.00187434 -0.02123374  0.01089688 -0.00658129]]
(9, 5)
[[ 0.00305441  0.0142473   0.01463216  0.00438051 -0.00790625]
 [-0.00446241  0.00538865 -0.01549335  0.00220945  0.00417095]
 [ 0.01768388  0.00480146 -0.01295475 -0.00036376  0.00895446]]


In [20]:
def softmax(x):
	"""ArithmeticError: Return softmax of x."""
	e_x = np.exp(x - np.max(x))
	return e_x / e_x.sum()

def forward(c, o, V, U):
	"""
	c: center word index
	o: context word index
	Returns: loss, probs, v_c
	"""
	v_c = V[c]  # (d,)
	scores = U @ v_c  # (V,)
	probs = softmax(scores)  # (V,)
	loss = -np.log(probs[o])  # scalar
	return loss, probs, v_c

def backward(c, o, probs, v_c, V, U):
	"""
	Returns:
		grad_v_c: (d,)
		grad_U: (V, d)
	"""
	vocab_size, embed_dim = U.shape

	grad_U = np.outer(probs, v_c)
	grad_U[o] -= v_c  # (V, d)

	grad_v_c = U.T @ probs - U[o]  # (d,)

	return grad_v_c, grad_U

def sgd_step(c, o, V, U, learning_rate=0.01):
	"""
	Perform one step of SGD for a single training pair (c, o)
	"""
	loss, probs, v_c = forward(c, o, V, U)
	grad_v_c, grad_U = backward(c, o, probs, v_c, V, U)

	# Update parameters
	V[c] -= learning_rate * grad_v_c
	U -= learning_rate * grad_U

	return loss

def train(token_ids, vocab_size, embed_dim, window_size, epochs, lr):
	V = np.random.randn(vocab_size, embed_dim) * 0.01
	U = np.random.randn(vocab_size, embed_dim) * 0.01

	pairs = generate_training_pairs(token_ids, window_size)

	losses = []
	for epoch in range(epochs):
		total_loss = 0
		np.random.shuffle(pairs) # shuffle training pairs

		for center, context in pairs:
			loss, probs, v_c = forward(center, context, V, U)
			grad_v_c, grad_U = backward(center, context, probs, v_c, V, U)

			# Update parameters
			V[center] -= lr * grad_v_c
			U -= lr * grad_U

			total_loss += loss

		avg_loss = total_loss / len(pairs)
		losses.append(avg_loss)
		if epoch % 10 == 0:
			print(f"Epoch {epoch}, Loss: {avg_loss}")

	return V, U, losses


def cosine_sim(a, b):
	"Compute cosine similarity between two vectors"
	return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def most_similar(word, word2idx, idx2word, V, top_n=5):
	"Find the most similar words to the given word"
	idx = word2idx[word]
	vec = V[idx]
	sims = [(cosine_sim(vec, V[i]), idx2word[i]) for i in range(len(V)) if i != idx]
	sims.sort(reverse=True)
	return sims[:top_n]

In [21]:
from comet_ml import Experiment

experiment = Experiment()

HP = {
    "vocab_size": vocab_size,
    "embed_dim": 5,
    "window_size": 2,
    "epochs": 1000,
    "lr": 0.01
}
experiment.log_parameters(HP)

V = np.random.randn(vocab_size, embed_dim) * 0.01
U = np.random.randn(vocab_size, embed_dim) * 0.01
pairs = generate_training_pairs(token_ids, window_size=HP["window_size"])

for epoch in range(HP["epochs"]):
	total_loss = 0
	np.random.shuffle(pairs)  # shuffle training pairs

	for center, context in pairs:
		loss, probs, v_c = forward(center, context, V, U)
		grad_v_c, grad_U = backward(center, context, probs, v_c, V, U)

		# Update parameters
		V[center] -= HP["lr"] * grad_v_c
		U -= HP["lr"] * grad_U

		total_loss += loss

	avg_loss = total_loss / len(pairs)
	experiment.log_metric("loss", avg_loss, step=epoch)
	if epoch % 10 == 0:
		print(f"Epoch {epoch}, Loss: {avg_loss}")


experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : stingy_lacquer_1985
COMET INFO:     url                   : https://www.comet.com/zdy23/general/afec65a9dcac47f398af67029eda1687
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [1000] : (1.5996809510856589, 2.1972085558567396)
COMET INFO:   Parameters:
COMET INFO:     embed_dim   : 5
COMET INFO:     epochs      : 1000
COMET INFO:     lr          : 0.01
COMET INFO:     vocab_size  : 9
COMET INFO:     window_size : 2
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (9.29 KB)
COMET INFO:

Epoch 0, Loss: 2.1972514338684164
Epoch 10, Loss: 2.197202662598001
Epoch 20, Loss: 2.197149510721689
Epoch 30, Loss: 2.1970766720322117
Epoch 40, Loss: 2.196962801444172
Epoch 50, Loss: 2.196773510966582
Epoch 60, Loss: 2.196450660154463
Epoch 70, Loss: 2.195893072294646
Epoch 80, Loss: 2.1949245593685647
Epoch 90, Loss: 2.1932429598986034
Epoch 100, Loss: 2.190333449955716
Epoch 110, Loss: 2.1853568187372465
Epoch 120, Loss: 2.1770288834186355
Epoch 130, Loss: 2.163629020740985
Epoch 140, Loss: 2.143440515638888
Epoch 150, Loss: 2.115868215317386
Epoch 160, Loss: 2.082825440254528
Epoch 170, Loss: 2.0484661834366316
Epoch 180, Loss: 2.016145867120559
Epoch 190, Loss: 1.9863825708452143
Epoch 200, Loss: 1.9577325561906238
Epoch 210, Loss: 1.9296515033027761
Epoch 220, Loss: 1.902986177123027
Epoch 230, Loss: 1.8795282602892596
Epoch 240, Loss: 1.860135856922659
Epoch 250, Loss: 1.8446283170086275
Epoch 260, Loss: 1.8318030588140515
Epoch 270, Loss: 1.8201755999128235
Epoch 280, Loss: 

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : wise_basis_6613
COMET INFO:     url                   : https://www.comet.com/zdy23/general/6b89cab700604db19c0c74b9cea63c99
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [1000] : (1.5949690122755855, 2.1972514338684164)
COMET INFO:   Parameters:
COMET INFO:     embed_dim   : 5
COMET INFO:     epochs      : 1000
COMET INFO:     lr          : 0.01
COMET INFO:     vocab_size  : 9
COMET INFO:     window_size : 2
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (9.29 KB)
COMET INFO:    

Epoch 830, Loss: 1.598665678203818
Epoch 840, Loss: 1.5986522009154291
Epoch 850, Loss: 1.5981621592426063
Epoch 860, Loss: 1.5979357225694104
Epoch 870, Loss: 1.5978719319575378
Epoch 880, Loss: 1.5974248939783462
Epoch 890, Loss: 1.597229984421019
Epoch 900, Loss: 1.5972502836294626
Epoch 910, Loss: 1.5967586109693739
Epoch 920, Loss: 1.5966868952941213
Epoch 930, Loss: 1.5965810183062876
Epoch 940, Loss: 1.5961801438594077
Epoch 950, Loss: 1.596249528813047
Epoch 960, Loss: 1.59606448846752
Epoch 970, Loss: 1.5957737591237635
Epoch 980, Loss: 1.5959408692276604
Epoch 990, Loss: 1.5956279622094776


COMET INFO: Please wait for metadata to finish uploading (timeout is 3600 seconds)
COMET INFO: Uploading 2244 metrics, params and output messages
